<a href="https://colab.research.google.com/github/umutbarandemir/CampusCam/blob/main/Kamp%C3%BCsFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
# ============================================================
# HÜCRE 1 — KURULUM
# ============================================================
import subprocess, sys

print("📦 Gerekli kütüphaneler kuruluyor... (1-2 dakika sürebilir)")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics>=8.0.0",
    "gradio>=4.0.0",
    "opencv-python-headless",
    "pandas",
    "matplotlib",
    "Pillow"
], check=True)
print("✅ Kurulum tamamlandı!")


# %%

📦 Gerekli kütüphaneler kuruluyor... (1-2 dakika sürebilir)
✅ Kurulum tamamlandı!


In [19]:
# ============================================================
# HÜCRE 2 — KÜTÜPHANELERİ İÇE AKTAR VE CİHAZ KONTROL
# ============================================================
import torch
import torchvision
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    FasterRCNN_ResNet50_FPN_Weights
)
from ultralytics import YOLO

import cv2
import numpy as np
import time
import pandas as pd
import gradio as gr
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

# Cihaz seçimi (GPU varsa GPU, yoksa CPU)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Kullanılan cihaz: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   ⚠️  GPU bulunamadı. CPU kullanılıyor — Faster R-CNN yavaş çalışacak.")
    print("   Colab'da: Çalışma Zamanı → Çalışma Zamanı Türünü Değiştir → T4 GPU")

print(f"\n📦 PyTorch: {torch.__version__}")
print(f"📦 Torchvision: {torchvision.__version__}")



🖥️  Kullanılan cihaz: cuda
   GPU: Tesla T4
   VRAM: 15.6 GB

📦 PyTorch: 2.11.0+cu128
📦 Torchvision: 0.26.0+cu128


In [20]:
# ============================================================
# HÜCRE 3 — COCO SINIF İSİMLERİ (80 sınıf)
# ============================================================
# Faster R-CNN 1-indexed kullanır: 0 = background, 1 = person, ...
COCO_CLASSES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane',
    'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant',
    'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse',
    'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack',
    'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis',
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass',
    'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple',
    'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed',
    'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote',
    'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink',
    'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear',
    'hair drier', 'toothbrush'
]

# Renk paleti (RGB formatı — Gradio için)
YOLO_COLOR_RGB  = (50, 205, 50)    # Lime Green
FRCNN_COLOR_RGB = (255, 140, 0)    # Dark Orange

# Renk paleti (BGR formatı — OpenCV için)
YOLO_COLOR_BGR  = (50, 205, 50)[::-1]
FRCNN_COLOR_BGR = (255, 140, 0)[::-1]

print(f"✅ {len(COCO_CLASSES) - 1} COCO sınıfı yüklendi")
print(f"   Kampüs için önemli sınıflar: person, bicycle, car, motorcycle, bus, backpack")

✅ 80 COCO sınıfı yüklendi
   Kampüs için önemli sınıflar: person, bicycle, car, motorcycle, bus, backpack


In [21]:
# ============================================================
# HÜCRE 4 — MODELLERİ YÜKLE
# ============================================================

# ---- YOLOv8n ----
print("⏳ YOLOv8n yükleniyor...")
yolo_model = YOLO('yolov8n.pt')   # 'n' = nano, en hızlı versiyon
yolo_model.to(DEVICE)
print("✅ YOLOv8n hazır!")

# ---- Faster R-CNN (ResNet-50 + FPN) ----
print("⏳ Faster R-CNN (ResNet-50-FPN) yükleniyor... (daha büyük model, biraz sürer)")
frcnn_weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
frcnn_model = fasterrcnn_resnet50_fpn(weights=frcnn_weights)
frcnn_model.to(DEVICE)
frcnn_model.eval()
frcnn_transforms = frcnn_weights.transforms()
print("✅ Faster R-CNN hazır!")

# ---- Model bilgisi ----
yolo_params  = sum(p.numel() for p in yolo_model.model.parameters())
frcnn_params = sum(p.numel() for p in frcnn_model.parameters())
print(f"\n📊 Model karşılaştırması:")
print(f"   YOLOv8n     : {yolo_params/1e6:.1f}M parametre | Single-stage | Anchor-free")
print(f"   Faster R-CNN: {frcnn_params/1e6:.1f}M parametre | Two-stage   | Anchor-based")


⏳ YOLOv8n yükleniyor...
✅ YOLOv8n hazır!
⏳ Faster R-CNN (ResNet-50-FPN) yükleniyor... (daha büyük model, biraz sürer)
✅ Faster R-CNN hazır!

📊 Model karşılaştırması:
   YOLOv8n     : 3.2M parametre | Single-stage | Anchor-free
   Faster R-CNN: 41.8M parametre | Two-stage   | Anchor-based


In [22]:
# ============================================================
# HÜCRE 5 — DETECTION FONKSİYONLARI
# ============================================================

def detect_yolo(image_rgb, conf_threshold=0.50, iou_threshold=0.45):
    """
    YOLOv8n ile nesne tespiti.

    Mimari notu:
    - Single-stage: backbone → neck (PAN) → head (tek geçiş)
    - Anchor-free: her grid cell için offset tahmin eder
    - NMS (Non-Maximum Suppression) iou_threshold parametresiyle kontrol edilir

    Args:
        image_rgb (np.ndarray): RGB formatında görüntü [H,W,3]
        conf_threshold (float): Minimum güven skoru (0.0–1.0)
        iou_threshold  (float): NMS IoU eşiği (0.0–1.0)

    Returns:
        boxes      : [[x1,y1,x2,y2], ...] piksel koordinatları
        scores     : [float, ...] güven skorları
        class_ids  : [int, ...]
        class_names: [str, ...]
        time_ms    : çıkarım süresi (ms)
    """
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()

    results = yolo_model(
        image_rgb,
        conf=conf_threshold,
        iou=iou_threshold,
        verbose=False
    )

    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    time_ms = (time.perf_counter() - t0) * 1000

    boxes, scores, class_ids, class_names = [], [], [], []
    for r in results:
        if r.boxes is None or len(r.boxes) == 0:
            continue
        for box in r.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            score   = float(box.conf[0].cpu())
            cls_id  = int(box.cls[0].cpu())
            cls_name = yolo_model.names[cls_id]
            boxes.append([x1, y1, x2, y2])
            scores.append(score)
            class_ids.append(cls_id)
            class_names.append(cls_name)

    return boxes, scores, class_ids, class_names, time_ms


def detect_frcnn(image_rgb, conf_threshold=0.50):
    """
    Faster R-CNN ile nesne tespiti.

    Mimari notu:
    - Stage 1 (RPN): Region Proposal Network — olası nesne bölgelerini üretir
    - Stage 2 (Head): Her bölge için sınıflandırma + kutu düzeltmesi yapar
    - FPN: Feature Pyramid Network — çok ölçekli özellik haritaları

    Neden iki aşama?
    - RPN object/background ayrımı yaparak yüksek recall'u yakalar
    - Head ise bu bölgelerde ince sınıflandırma yapar → daha yüksek precision

    Args:
        image_rgb     (np.ndarray): RGB formatında görüntü [H,W,3]
        conf_threshold (float)    : Minimum güven skoru

    Returns:
        boxes, scores, class_ids, class_names, time_ms
    """
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()

    pil_img = Image.fromarray(image_rgb)
    img_tensor = frcnn_transforms(pil_img).to(DEVICE)

    with torch.no_grad():
        predictions = frcnn_model([img_tensor])

    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    time_ms = (time.perf_counter() - t0) * 1000

    pred = predictions[0]
    keep = pred['scores'] > conf_threshold

    raw_boxes  = pred['boxes'][keep].cpu().numpy()
    raw_scores = pred['scores'][keep].cpu().numpy()
    raw_labels = pred['labels'][keep].cpu().numpy().astype(int)

    boxes, scores, class_ids, class_names = [], [], [], []
    for box, score, label in zip(raw_boxes, raw_scores, raw_labels):
        if label >= len(COCO_CLASSES):
            continue
        cls_name = COCO_CLASSES[label]
        if cls_name in ('__background__', 'N/A'):
            continue
        boxes.append(list(map(int, box)))
        scores.append(float(score))
        class_ids.append(int(label))
        class_names.append(cls_name)

    return boxes, scores, class_ids, class_names, time_ms


def draw_detections(image_rgb, boxes, scores, class_names, color_rgb, model_label):
    """
    Görüntü üzerine bounding box + etiket çiz.
    Hem YOLOv8 hem Faster R-CNN sonuçları için kullanılır.

    Args:
        image_rgb   : RGB numpy array
        boxes       : [[x1,y1,x2,y2], ...]
        scores      : [float, ...]
        class_names : [str, ...]
        color_rgb   : (R, G, B) tuple
        model_label : Görüntü başlığı için model adı

    Returns:
        Annotated görüntü (RGB numpy array)
    """
    img = image_rgb.copy()
    h, w = img.shape[:2]

    # Görüntü boyutuna göre dinamik font/kalınlık
    font_scale = max(0.40, min(0.90, w / 1200))
    thickness  = max(1, int(w / 600))
    color_bgr  = color_rgb[::-1]   # PIL/matplotlib RGB → OpenCV BGR

    # OpenCV BGR'ye çevir (draw için)
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

    for box, score, cls_name in zip(boxes, scores, class_names):
        x1, y1, x2, y2 = box

        # Bounding box
        cv2.rectangle(img_bgr, (x1, y1), (x2, y2), color_bgr, thickness + 1)

        # Etiket
        label_text = f"{cls_name}: {score:.2f}"
        (tw, th), baseline = cv2.getTextSize(
            label_text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness
        )
        label_y = max(y1 - 4, th + baseline + 4)

        # Etiket arka planı (okunabilirlik için)
        cv2.rectangle(
            img_bgr,
            (x1, label_y - th - baseline - 2),
            (x1 + tw + 6, label_y + baseline),
            color_bgr, -1
        )
        # Etiket yazısı (renk parlaklığına göre siyah/beyaz)
        brightness = 0.299*color_rgb[0] + 0.587*color_rgb[1] + 0.114*color_rgb[2]
        text_color = (0, 0, 0) if brightness > 128 else (255, 255, 255)
        cv2.putText(
            img_bgr, label_text, (x1 + 3, label_y - 2),
            cv2.FONT_HERSHEY_SIMPLEX, font_scale, text_color, thickness, cv2.LINE_AA
        )

    # Model başlığı (sol üst köşe)
    header = f"{model_label}  |  {len(boxes)} nesne"
    cv2.putText(
        img_bgr, header, (10, 32),
        cv2.FONT_HERSHEY_SIMPLEX, 0.8, color_bgr, 2, cv2.LINE_AA
    )

    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)



In [23]:
# ============================================================
# HÜCRE 6 — KARŞILAŞTIRMA, OTOMATİK ANALİZ VE METRİK
# ============================================================

def generate_analysis(yolo_boxes, yolo_scores, yolo_names, yolo_ms,
                      frcnn_boxes, frcnn_scores, frcnn_names, frcnn_ms,
                      image_shape, conf_threshold):
    """
    Model çıktılarını inceleyerek otomatik teknik yorum üretir.
    Tespit sayısı, bbox boyutları, güven skorları ve sınıf dağılımı
    üzerinden olası failure case kategorisini tahmin eder.
    """
    h, w = image_shape[:2]
    y_n, f_n = len(yolo_boxes), len(frcnn_boxes)
    lines = ["## 🔬 Otomatik Teknik Analiz", ""]

    # ── 1. Tespit karşılaştırması ─────────────────────────────
    lines.append("### 📊 Tespit Karşılaştırması")
    diff = y_n - f_n
    if diff == 0:
        lines.append(f"Her iki model de **{y_n}** nesne tespit etti.")
    elif diff > 0:
        lines.append(f"YOLOv8n **{y_n}** nesne tespit ederken Faster R-CNN **{f_n}** tespit etti "
                     f"(**{abs(diff)} fazla**).")
        lines.append("> Anchor-free grid ataması kalabalık sahnelerde daha fazla öneri üretir; "
                     "ancak bu FP riskini artırabilir.")
    else:
        lines.append(f"Faster R-CNN **{f_n}** nesne tespit ederken YOLOv8n **{y_n}** tespit etti "
                     f"(**{abs(diff)} fazla**).")
        lines.append("> RPN Stage-1, YOLOv8\'ın tek geçişte kaçırdığı bazı bölgeleri yakalayabilir.")
    lines.append("")

    # ── 2. Hız analizi ────────────────────────────────────────
    lines.append("### ⚡ Hız Analizi")
    ratio = frcnn_ms / yolo_ms if yolo_ms > 0 else 0
    y_fps = 1000 / yolo_ms if yolo_ms > 0 else 0
    f_fps = 1000 / frcnn_ms if frcnn_ms > 0 else 0
    lines.append(f"YOLOv8n: **{yolo_ms:.1f} ms** ({y_fps:.1f} FPS)  |  "
                 f"Faster R-CNN: **{frcnn_ms:.1f} ms** ({f_fps:.1f} FPS)")
    lines.append(f"> YOLOv8n, bu görüntüde Faster R-CNN\'den **{ratio:.1f}×** daha hızlı. "
                 "Fark, RPN\'in ek hesaplama yükünden kaynaklanmaktadır.")
    lines.append("")

    # ── 3. Küçük nesne analizi ────────────────────────────────
    lines.append("### 🔭 Küçük Nesne Analizi")
    SMALL = 32 * 32   # 32×32 piksel altı = küçük nesne (COCO standardı)
    y_small = sum(1 for b in yolo_boxes  if (b[2]-b[0])*(b[3]-b[1]) < SMALL)
    f_small = sum(1 for b in frcnn_boxes if (b[2]-b[0])*(b[3]-b[1]) < SMALL)
    if y_small > 0 or f_small > 0:
        lines.append(f"⚠️ Küçük nesne (< 32×32 px): YOLOv8n → **{y_small}** | Faster R-CNN → **{f_small}**")
        lines.append("> Stride-32 özellik haritasında bu boyuttaki nesneler yalnızca ~1 hücreye düşer. "
                     "Özellik kaybı → düşük güven skoru veya tamamen kaçırma.")
    else:
        lines.append("✅ Küçük nesne sorunu gözlemlenmedi (tüm bbox\'lar ≥ 32×32 piksel).")
    lines.append("")

    # ── 4. Güven skoru analizi ────────────────────────────────
    lines.append("### 📈 Güven Skoru Analizi")
    if yolo_scores and frcnn_scores:
        y_avg, f_avg = np.mean(yolo_scores), np.mean(frcnn_scores)
        y_low = sum(1 for s in yolo_scores  if conf_threshold <= s < conf_threshold + 0.15)
        f_low = sum(1 for s in frcnn_scores if conf_threshold <= s < conf_threshold + 0.15)
        lines.append(f"Ort. güven → YOLOv8n: **{y_avg:.3f}**  |  Faster R-CNN: **{f_avg:.3f}**")
        if y_low:
            lines.append(f"⚠️ YOLOv8n\'de **{y_low}** tespit eşiğin hemen üzerinde → potansiyel yanlış pozitif (FP).")
        if f_low:
            lines.append(f"⚠️ Faster R-CNN\'de **{f_low}** tespit eşiğin hemen üzerinde → potansiyel yanlış pozitif (FP).")
    elif not yolo_scores and not frcnn_scores:
        lines.append(f"⚠️ Her iki model de **{conf_threshold:.2f}** eşiğinde tespit üretemedi.")
        lines.append(f"> Eşiği **{max(0.10, conf_threshold-0.15):.2f}\'e** düşürmeyi deneyin. "
                     "Gece/düşük ışık veya küçük nesne senaryolarında bu beklenen bir davranıştır.")
    elif not yolo_scores:
        lines.append(f"⚠️ YOLOv8n bu eşikte tespit üretemedi; Faster R-CNN {f_n} nesne buldu.")
    elif not frcnn_scores:
        lines.append(f"⚠️ Faster R-CNN bu eşikte tespit üretemedi; YOLOv8n {y_n} nesne buldu.")
    lines.append("")

    # ── 5. Kalabalık sahne ────────────────────────────────────
    y_persons = yolo_names.count("person")
    if y_persons >= 4:
        lines.append("### 👥 Kalabalık Sahne Tespiti")
        lines.append(f"**{y_persons}** kişi tespit edildi → kalabalık sahne kategorisi.")
        lines.append("> Üst üste binen bounding box\'lar (oklüzyon) NMS tarafından bastırılıyor olabilir. "
                     "**IoU eşiğini 0.35\'e** düşürerek NMS\'i daha toleranslı yapabilirsiniz.")
        lines.append("")

    # ── 6. Olası failure case tahmini ────────────────────────
    lines.append("### ⚠️ Olası Failure Case Kategorisi")
    hints = []
    if y_small > 1 or f_small > 1:
        hints.append("**FC-2 (Küçük Nesne):** Tespit edilen küçük bbox\'lar stride kısıtına işaret ediyor.")
    if y_persons >= 4:
        hints.append("**FC-1 (Oklüzyon):** Kalabalık sahne — arkadaki kişiler NMS\'de bastırılmış olabilir.")
    if yolo_scores and np.mean(yolo_scores) < 0.45:
        hints.append("**FC-4 (Düşük Işık / Domain Shift):** Düşük güven skorları dağılım kaymasını işaret edebilir.")
    if abs(diff) >= 3:
        hints.append("**FC-3 / FC-5:** Modeller arası belirgin fark; hareket bulanıklığı veya arka plan "
                     "karışıklığına işaret edebilir.")
    if hints:
        for h in hints:
            lines.append(f"- {h}")
    else:
        lines.append("✅ Bu görüntüde belirgin bir failure case kategorisi tespit edilmedi — normal performans aralığı.")

    return "\n".join(lines)


def compare_models(input_image, conf_threshold=0.50, iou_threshold=0.45):
    """
    Her iki modeli aynı görüntü üzerinde çalıştırır, karşılaştırır
    ve otomatik teknik analiz metni üretir.

    Returns:
        yolo_img    : YOLOv8 sonucu (RGB numpy)
        frcnn_img   : Faster R-CNN sonucu (RGB numpy)
        metrics_df  : Karşılaştırma tablosu (pandas DataFrame)
        analysis_md : Otomatik teknik analiz (Markdown string)
    """
    EMPTY = pd.DataFrame({"Uyarı": ["Lütfen bir görüntü yükleyin."]})
    if input_image is None:
        return None, None, EMPTY, "⚠️ Görüntü yüklenmedi."

    image_rgb = input_image if isinstance(input_image, np.ndarray) else np.array(input_image)

    # ── YOLOv8 ──────────────────────────────────────────────
    yolo_boxes, yolo_scores, _, yolo_names, yolo_ms = detect_yolo(
        image_rgb, conf_threshold, iou_threshold)
    yolo_img = draw_detections(image_rgb, yolo_boxes, yolo_scores,
                               yolo_names, YOLO_COLOR_RGB, "YOLOv8n")

    # ── Faster R-CNN ─────────────────────────────────────────
    frcnn_boxes, frcnn_scores, _, frcnn_names, frcnn_ms = detect_frcnn(
        image_rgb, conf_threshold)
    frcnn_img = draw_detections(image_rgb, frcnn_boxes, frcnn_scores,
                                frcnn_names, FRCNN_COLOR_RGB, "Faster R-CNN")

    # ── Metrik tablosu ───────────────────────────────────────
    def by_class(names):
        c = {}
        for n in names: c[n] = c.get(n, 0) + 1
        return c

    yc, fc = by_class(yolo_names), by_class(frcnn_names)
    all_cls = sorted(set(list(yc) + list(fc)))

    rows = [
        {"Metrik": "⚡ Toplam Tespit",
         "YOLOv8n": len(yolo_boxes), "Faster R-CNN": len(frcnn_boxes)},
        {"Metrik": "⏱️ Çıkarım Süresi (ms)",
         "YOLOv8n": f"{yolo_ms:.1f}", "Faster R-CNN": f"{frcnn_ms:.1f}"},
        {"Metrik": "🚀 Anlık FPS",
         "YOLOv8n": f"{1000/yolo_ms:.1f}" if yolo_ms>0 else "—",
         "Faster R-CNN": f"{1000/frcnn_ms:.1f}" if frcnn_ms>0 else "—"},
        {"Metrik": "📊 Ort. Güven Skoru",
         "YOLOv8n": f"{np.mean(yolo_scores):.3f}" if yolo_scores else "—",
         "Faster R-CNN": f"{np.mean(frcnn_scores):.3f}" if frcnn_scores else "—"},
        {"Metrik": "─── Sınıf Bazlı ───", "YOLOv8n": "", "Faster R-CNN": ""},
    ]
    for cls in all_cls:
        rows.append({"Metrik": f"  {cls}",
                     "YOLOv8n": yc.get(cls,0), "Faster R-CNN": fc.get(cls,0)})

    # ── Otomatik analiz ──────────────────────────────────────
    analysis = generate_analysis(
        yolo_boxes, yolo_scores, yolo_names, yolo_ms,
        frcnn_boxes, frcnn_scores, frcnn_names, frcnn_ms,
        image_rgb.shape, conf_threshold
    )

    return yolo_img, frcnn_img, pd.DataFrame(rows), analysis


def fps_benchmark(input_image, n_runs=20):
    """
    N çalıştırma üzerinden ortalama FPS ölçer (warmup hariç).
    """
    if input_image is None:
        return pd.DataFrame({"Uyarı": ["Görüntü yükleyin."]})

    image_rgb = input_image if isinstance(input_image, np.ndarray) else np.array(input_image)
    print(f"🔄 Benchmark başladı ({n_runs} çalıştırma)...")

    detect_yolo(image_rgb)   # warmup
    detect_frcnn(image_rgb)

    yt, ft = [], []
    for i in range(n_runs):
        _, _, _, _, y = detect_yolo(image_rgb)
        _, _, _, _, f = detect_frcnn(image_rgb)
        yt.append(y); ft.append(f)
        if (i+1) % 5 == 0:
            print(f"   {i+1}/{n_runs} tamamlandı...")

    df = pd.DataFrame({
        "Model":            ["YOLOv8n", "Faster R-CNN"],
        "Ort. Süre (ms)":  [f"{np.mean(yt):.2f}", f"{np.mean(ft):.2f}"],
        "Std Sapma (ms)":  [f"{np.std(yt):.2f}",  f"{np.std(ft):.2f}"],
        "Min (ms)":        [f"{np.min(yt):.2f}",  f"{np.min(ft):.2f}"],
        "Max (ms)":        [f"{np.max(yt):.2f}",  f"{np.max(ft):.2f}"],
        "Ort. FPS":        [f"{1000/np.mean(yt):.1f}", f"{1000/np.mean(ft):.1f}"],
        "Hız Oranı":       [f"{np.mean(ft)/np.mean(yt):.1f}× daha hızlı", "—"],
    })
    print("\n📊 Sonuçlar:")
    print(df.to_string(index=False))
    return df


In [24]:
# ============================================================
# HÜCRE 7 — VİDEO İŞLEME VE FPS GRAFİĞİ
# ============================================================

def create_fps_graph(yolo_times, frcnn_times, src_fps=25.0):
    """
    Kare bazlı çıkarım süresi ve FPS grafiği üretir.
    Matplotlib ile çizilir, RGB numpy array olarak döner
    (Gradio gr.Image bileşeninde gösterilmek üzere).
    """
    frames = list(range(len(yolo_times)))
    y_fps  = [1000/t for t in yolo_times]
    f_fps  = [1000/t for t in frcnn_times]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), dpi=110)
    fig.patch.set_facecolor("#F8FAFC")

    # ── Çıkarım süresi (ms) ─────────────────────────────────
    ax1.set_facecolor("#FFFFFF")
    ax1.plot(frames, yolo_times,  color="#32CD32", lw=1.8, label="YOLOv8n",      alpha=0.85)
    ax1.plot(frames, frcnn_times, color="#FF8C00", lw=1.8, label="Faster R-CNN", alpha=0.85)
    ax1.axhline(40, color="red", ls="--", lw=1.2, alpha=0.6, label="25 FPS eşiği (40 ms)")
    ax1.fill_between(frames, yolo_times,  alpha=0.10, color="#32CD32")
    ax1.fill_between(frames, frcnn_times, alpha=0.10, color="#FF8C00")
    ax1.set_ylabel("Çıkarım Süresi (ms)", fontsize=11)
    ax1.set_title("Kare Bazlı Çıkarım Süresi Karşılaştırması", fontsize=13, fontweight="bold")
    ax1.legend(fontsize=10); ax1.grid(True, alpha=0.3)
    ax1.set_xlim(0, max(1, len(frames)-1))

    # Özet istatistikler
    ax1.text(0.01, 0.97,
             f"YOLOv8n ort: {np.mean(yolo_times):.1f} ms  |  "
             f"Faster R-CNN ort: {np.mean(frcnn_times):.1f} ms  |  "
             f"Hız farkı: {np.mean(frcnn_times)/np.mean(yolo_times):.1f}×",
             transform=ax1.transAxes, fontsize=9, va="top",
             bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))

    # ── FPS ──────────────────────────────────────────────────
    ax2.set_facecolor("#FFFFFF")
    ax2.plot(frames, y_fps,  color="#32CD32", lw=1.8, label="YOLOv8n",      alpha=0.85)
    ax2.plot(frames, f_fps,  color="#FF8C00", lw=1.8, label="Faster R-CNN", alpha=0.85)
    ax2.axhline(25, color="red",   ls="--", lw=1.2, alpha=0.6, label="Gerçek zamanlı eşiği (25 FPS)")
    ax2.axhline(60, color="green", ls=":",  lw=1.0, alpha=0.4, label="Hedef: 60 FPS")
    ax2.fill_between(frames, y_fps,  alpha=0.10, color="#32CD32")
    ax2.fill_between(frames, f_fps,  alpha=0.10, color="#FF8C00")
    ax2.set_xlabel("Kare Numarası", fontsize=11)
    ax2.set_ylabel("FPS", fontsize=11)
    ax2.set_title("Kare Bazlı FPS Karşılaştırması", fontsize=13, fontweight="bold")
    ax2.legend(fontsize=10); ax2.grid(True, alpha=0.3)
    ax2.set_xlim(0, max(1, len(frames)-1))
    ax2.set_ylim(bottom=0)

    plt.tight_layout(pad=2.0)

    import io
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=110, bbox_inches='tight')
    buf.seek(0)
    img = cv2.imdecode(np.frombuffer(buf.read(), dtype=np.uint8), cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.close(fig)
    return img


def process_video(video_path, conf_threshold=0.50, iou_threshold=0.45,
                  frame_skip=2, max_seconds=15):
    """
    Video üzerinde her iki modeli kare kare çalıştırır.
    Annotated video + FPS grafiği + istatistik tablosu üretir.

    Args:
        video_path    : Gradio\'nun geçici dosya yolu
        conf_threshold: Güven eşiği (0–1)
        iou_threshold : IoU eşiği — NMS (0–1)
        frame_skip    : Her kaçıncı kare işlenecek (1=hepsi, 2=her 2.)
        max_seconds   : İşlenecek maks. video süresi (saniye)

    Returns:
        out_yolo  : YOLOv8 annotated video yolu (H.264/mp4)
        out_frcnn : Faster R-CNN annotated video yolu
        df        : İstatistik tablosu
        fps_graph : Kare bazlı FPS grafiği (RGB numpy)
    """
    EMPTY_DF = pd.DataFrame({"Uyarı": ["Video yükleyin."]})

    if video_path is None:
        return None, None, EMPTY_DF, None

    # Gradio bazen dosya nesnesi gönderir
    if hasattr(video_path, "name"):
        video_path = video_path.name

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None, None, pd.DataFrame({"Uyarı": ["Video açılamadı."]}), None

    src_fps    = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    max_frames = int(src_fps * max_seconds)
    out_fps    = max(1.0, src_fps / frame_skip)

    print(f"🎬 Video: {width}×{height} @ {src_fps:.0f} FPS")
    print(f"   Ayarlar: her {frame_skip}. kare | maks {max_seconds}s → ~{max_frames} kare")

    tmp_y, tmp_f = "/tmp/yolo_raw.mp4", "/tmp/frcnn_raw.mp4"
    out_y, out_f = "/tmp/yolo_out.mp4", "/tmp/frcnn_out.mp4"

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    wy = cv2.VideoWriter(tmp_y, fourcc, out_fps, (width, height))
    wf = cv2.VideoWriter(tmp_f, fourcc, out_fps, (width, height))

    yolo_times, frcnn_times = [], []
    yolo_counts, frcnn_counts = [], []
    frame_idx = written = 0

    while True:
        ret, frame = cap.read()
        if not ret or frame_idx >= max_frames:
            break

        if frame_idx % frame_skip != 0:
            frame_idx += 1
            continue

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # YOLOv8
        yb, ys, _, yn, yms = detect_yolo(rgb, conf_threshold, iou_threshold)
        ya = draw_detections(rgb, yb, ys, yn, YOLO_COLOR_RGB,
                             f"YOLOv8n | #{frame_idx} | {len(yb)} nesne")
        wy.write(cv2.cvtColor(ya, cv2.COLOR_RGB2BGR))
        yolo_times.append(yms); yolo_counts.append(len(yb))

        # Faster R-CNN
        fb, fs, _, fn, fms = detect_frcnn(rgb, conf_threshold)
        fa = draw_detections(rgb, fb, fs, fn, FRCNN_COLOR_RGB,
                             f"Faster R-CNN | #{frame_idx} | {len(fb)} nesne")
        wf.write(cv2.cvtColor(fa, cv2.COLOR_RGB2BGR))
        frcnn_times.append(fms); frcnn_counts.append(len(fb))

        written += 1; frame_idx += 1
        if written % 15 == 0:
            print(f"   ⏳ {written} kare işlendi ({frame_idx/src_fps:.1f}s / {max_seconds}s)")

    cap.release(); wy.release(); wf.release()
    print(f"✅ {written} kare işlendi. Yeniden kodlanıyor...")

    # FFmpeg H.264 yeniden kodlama (tarayıcı uyumluluğu)
    r1 = os.system(f"ffmpeg -y -i {tmp_y} -c:v libx264 -pix_fmt yuv420p -movflags +faststart {out_y} -loglevel error")
    r2 = os.system(f"ffmpeg -y -i {tmp_f} -c:v libx264 -pix_fmt yuv420p -movflags +faststart {out_f} -loglevel error")
    if r1 != 0 or r2 != 0:
        print("⚠️ FFmpeg H.264 başarısız — ham dosya kullanılıyor")
        out_y, out_f = tmp_y, tmp_f
    print("✅ Video hazır!")

    if not yolo_times:
        return None, None, pd.DataFrame({"Uyarı": ["Kare işlenemedi."]}), None

    # Metrik tablosu
    rows = [
        {"Metrik": "🎞️ İşlenen Kare",
         "YOLOv8n": written, "Faster R-CNN": written},
        {"Metrik": "⏱️ Ort. Çıkarım (ms)",
         "YOLOv8n": f"{np.mean(yolo_times):.1f}",
         "Faster R-CNN": f"{np.mean(frcnn_times):.1f}"},
        {"Metrik": "🚀 Ort. FPS (çıkarım)",
         "YOLOv8n": f"{1000/np.mean(yolo_times):.1f}",
         "Faster R-CNN": f"{1000/np.mean(frcnn_times):.1f}"},
        {"Metrik": "📦 Ort. Tespit / Kare",
         "YOLOv8n": f"{np.mean(yolo_counts):.1f}",
         "Faster R-CNN": f"{np.mean(frcnn_counts):.1f}"},
        {"Metrik": "📦 Max Tespit (tek karede)",
         "YOLOv8n": max(yolo_counts), "Faster R-CNN": max(frcnn_counts)},
        {"Metrik": "📦 Min Tespit (en boş kare)",
         "YOLOv8n": min(yolo_counts), "Faster R-CNN": min(frcnn_counts)},
        {"Metrik": "⚡ Hız Avantajı",
         "YOLOv8n": f"{np.mean(frcnn_times)/np.mean(yolo_times):.1f}× daha hızlı",
         "Faster R-CNN": "—"},
        {"Metrik": "📹 İşlenen Video Süresi",
         "YOLOv8n": f"{written*frame_skip/src_fps:.1f}s",
         "Faster R-CNN": f"{written*frame_skip/src_fps:.1f}s"},
    ]

    fps_graph = create_fps_graph(yolo_times, frcnn_times, src_fps)
    return out_y, out_f, pd.DataFrame(rows), fps_graph


In [25]:
# ============================================================
# HÜCRE 8 — GRADIO KULLANICI ARAYÜZÜ  (v3 — düzeltilmiş)
# ============================================================
# Değişiklikler:
#   • Görüntü sekmeleri: sources=["upload","webcam"] (snapshot, video değil)
#   • Video sekmesi: sources=["upload"] — yalnızca dosya yükleme
#   • compare_models 4 çıktı döndürüyor → analiz metni de gösteriliyor
#   • process_video 4 çıktı döndürüyor → FPS grafiği de gösteriliyor
#   • Benchmark sekmesi: sources=["upload"] (webcam kaldırıldı)
#   • Failure case: sources=["upload","webcam"] korundu
# ============================================================

def build_and_launch():
    with gr.Blocks(title="Kampüs Güvenlik Sistemi", theme=gr.themes.Soft()) as demo:

        gr.Markdown("""
        # 🎓 Kampüs Güvenlik Kamerası — Object Detection Karşılaştırma Sistemi
        **OMÜ Yüksek Lisans | Bilgisayarlı Görü Dersi Final Projesi**
        **YOLOv8n** (yeşil) vs **Faster R-CNN** (turuncu) — COCO pre-trained
        """)

        # ── SEKME 1: Görüntü / Webcam Karşılaştırma ─────────────
        with gr.Tab("📷 Görüntü / Webcam"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### ⚙️ Giriş & Ayarlar")
                    # sources=["upload","webcam"] → yükleme VEYA kameradan tek kare
                    input_img = gr.Image(
                        label="Görüntü Yükle veya Webcam Snapshot 📸",
                        type="numpy",
                        sources=["upload", "webcam"],
                        height=270
                    )
                    conf_s = gr.Slider(0.10, 0.90, 0.50, step=0.05,
                                       label="Güven Eşiği (Confidence Threshold)")
                    iou_s  = gr.Slider(0.10, 0.90, 0.45, step=0.05,
                                       label="IoU Eşiği — NMS ayarı")
                    run_btn = gr.Button("🔍 Her İki Modeli Çalıştır",
                                        variant="primary", size="lg")
                    gr.Markdown("""
                    **💡 Rehber:**
                    - Gündüz/net → güven: 0.45–0.55
                    - Kalabalık → IoU: 0.35–0.45
                    - Gece/düşük ışık → güven: 0.25–0.35
                    - Uzak mesafe → güven: 0.20–0.30
                    - 📸 Webcam: kamera ikonu → fotoğraf çek → analiz
                    """)

                with gr.Column(scale=2):
                    gr.Markdown("### 📊 Tespit Sonuçları")
                    with gr.Row():
                        yolo_out  = gr.Image(label="⚡ YOLOv8n — Yeşil",   height=300)
                        frcnn_out = gr.Image(label="🎯 Faster R-CNN — Turuncu", height=300)
                    metrics_tbl = gr.DataFrame(label="📈 Metrikler", wrap=True)
                    analysis_md = gr.Markdown(label="🔬 Otomatik Teknik Analiz")

            run_btn.click(
                fn=compare_models,
                inputs=[input_img, conf_s, iou_s],
                outputs=[yolo_out, frcnn_out, metrics_tbl, analysis_md]
            )

        # ── SEKME 2: Video Analizi ────────────────────────────────
        with gr.Tab("🎬 Video Analizi"):
            gr.Markdown("""
            ## Video Üzerinde Karşılaştırmalı Nesne Tespiti
            Video yükleyin → her kare her iki modelle işlenir → annotated video + FPS grafiği üretilir.
            > ⚠️ **Süre tahmini:** GPU ile her kare ~100–150ms. 15s/25fps video ≈ 1–2 dk.
            > Uzun videolarda **Kare Atlama ≥ 2** veya **Maks. Süre = 10s** öneririz.
            """)
            with gr.Row():
                with gr.Column(scale=1):
                    # sources=["upload"] → SADECE dosya yükleme, webcam/kayıt YOK
                    video_in = gr.Video(
                        label="📹 Video Yükle (.mp4 / .avi / .mov / .mkv)",
                        sources=["upload"]
                    )
                    v_conf = gr.Slider(0.10, 0.90, 0.45, step=0.05, label="Güven Eşiği")
                    v_iou  = gr.Slider(0.10, 0.90, 0.45, step=0.05, label="IoU Eşiği (NMS)")
                    v_skip = gr.Slider(1, 5, 2, step=1,
                                       label="Kare Atlama  (1=her kare | 2=her 2. kare, 2× hızlı)")
                    v_maxs = gr.Slider(5, 60, 15, step=5,
                                       label="Maks. İşlenecek Süre (saniye)")
                    vid_btn = gr.Button("🎬 Videoyu İşle", variant="primary", size="lg")
                    gr.Markdown("""
                    **💡 Hız/kalite dengesi:**
                    - Skip=1 → en yüksek kalite, yavaş
                    - Skip=2 → 2× hızlı, görsel kalite yeterli ✓
                    - Skip=3 → 3× hızlı, hafif titreme
                    """)
                with gr.Column(scale=1):
                    gr.Markdown("""
                    ### ▶️ Çıktılar
                    İşlem tamamlandığında:
                    1. İki annotated video (YOLO ve FRCNN)
                    2. Kare bazlı FPS grafiği
                    3. İstatistik tablosu
                    """)

            gr.Markdown("---")
            with gr.Row():
                vid_yolo_out  = gr.Video(label="⚡ YOLOv8n Sonucu",      height=330)
                vid_frcnn_out = gr.Video(label="🎯 Faster R-CNN Sonucu", height=330)
            fps_graph_out = gr.Image(label="📊 Kare Bazlı FPS Grafiği", height=350)
            vid_metrics   = gr.DataFrame(label="📊 Video İstatistikleri", wrap=True)

            vid_btn.click(
                fn=process_video,
                inputs=[video_in, v_conf, v_iou, v_skip, v_maxs],
                outputs=[vid_yolo_out, vid_frcnn_out, vid_metrics, fps_graph_out]
            )

        # ── SEKME 3: Failure Case Analizi ────────────────────────
        with gr.Tab("❌ Failure Case Analizi"):
            gr.Markdown("""
            ## Başarısız Durum Analizi
            Düşük güven eşiği ile modellerin hatalı tespitlerini (FP/FN) ortaya çıkarın.
            Sistem otomatik olarak hangi failure case kategorisine uyduğunu yorumlayacaktır.
            """)
            with gr.Row():
                # Failure case için webcam mantıklı: sahada çekilen görüntüleri test edebilir
                fail_img  = gr.Image(
                    label="Görüntü Yükle (oklüzyon/gece/küçük nesne vb.)",
                    type="numpy",
                    sources=["upload", "webcam"]
                )
                fail_conf = gr.Slider(0.10, 0.90, 0.25, step=0.05,
                                      label="Güven Eşiği  (FP görmek için düşük tutun: 0.20–0.35)")
            fail_btn = gr.Button("🔍 Failure Case Analiz Et", variant="secondary")
            with gr.Row():
                fail_yolo  = gr.Image(label="YOLOv8n Sonucu")
                fail_frcnn = gr.Image(label="Faster R-CNN Sonucu")
            fail_metrics = gr.DataFrame(label="Metrikler")
            fail_analysis = gr.Markdown(label="🔬 Otomatik Teknik Analiz")

            gr.Markdown("""
            ---
            ### 📋 Bilinen Failure Case Kategorileri

            | # | Senaryo | YOLOv8n | Faster R-CNN | Teknik Neden |
            |---|---------|---------|--------------|--------------|
            | FC-1 | Oklüzyon | FN ↑ (arka kayıp) | NMS bastırır | IoU > NMS eşiği |
            | FC-2 | Küçük nesne | Recall < 0.4 | FPN P2 kısmen yardımcı | Stride-32 özellik kaybı |
            | FC-3 | Hareket bulanıklığı | FP artışı | Güven düşer | Domain shift (COCO=net) |
            | FC-4 | Gece/düşük ışık | Recall düşer | Daha da zayıf | ImageNet norm. kayması |
            | FC-5 | Benzer renk arka plan | FP (sütun→kişi) | RPN proposal üretemez | Renk/doku kontrast kaybı |
            """)

            fail_btn.click(
                fn=lambda img, conf: compare_models(img, conf, 0.45),
                inputs=[fail_img, fail_conf],
                outputs=[fail_yolo, fail_frcnn, fail_metrics, fail_analysis]
            )

        # ── SEKME 4: FPS Benchmark ────────────────────────────────
        with gr.Tab("⚡ FPS Benchmark"):
            gr.Markdown("""
            ## Hız Karşılaştırması
            Aynı görüntüyü 20 kez çalıştırır; ortalama/std/min/max FPS ölçer.
            İlk 2 çalıştırma warmup olarak istatistiğe dahil edilmez.
            """)
            # Benchmark için SADECE dosya yükleme — webcam snapshot gereksiz
            bench_img = gr.Image(
                label="Benchmark Görüntüsü",
                type="numpy",
                sources=["upload"],
                height=260
            )
            bench_btn = gr.Button("🏁 Benchmark Başlat (20 çalıştırma)", variant="secondary")
            bench_out = gr.DataFrame(label="📊 Benchmark Sonuçları")

            bench_btn.click(
                fn=lambda img: fps_benchmark(img, 20) if img is not None
                               else pd.DataFrame({"Uyarı": ["Görüntü yükleyin."]}),
                inputs=[bench_img],
                outputs=[bench_out]
            )

        # ── SEKME 5: Model Bilgisi ────────────────────────────────
        with gr.Tab("ℹ️ Model Bilgisi"):
            gpu_name = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "Yok"
            gr.Markdown(f"""
            ## Yüklü Modeller ve Teknik Detaylar

            ### ⚡ YOLOv8n
            | Özellik | Değer |
            |---------|-------|
            | Mimari | Single-stage, anchor-free, CSPNet backbone |
            | Parametre | ~{sum(p.numel() for p in yolo_model.model.parameters())/1e6:.1f}M |
            | Sınıf | 80 (COCO) |
            | Avantaj | Hız, gerçek zamanlı |
            | Dezavantaj | Küçük / üst üste nesneler |

            ### 🎯 Faster R-CNN (ResNet-50-FPN)
            | Özellik | Değer |
            |---------|-------|
            | Mimari | Two-stage: RPN + ROI Head, anchor-based |
            | Parametre | ~{sum(p.numel() for p in frcnn_model.parameters())/1e6:.1f}M |
            | Sınıf | 80 (COCO) |
            | Avantaj | Yüksek doğruluk, oklüzyon |
            | Dezavantaj | Yavaş (iki aşamalı) |

            ### 🖥️ Ortam
            | | |
            |--|--|
            | Cihaz | {DEVICE} |
            | GPU | {gpu_name} |
            | PyTorch | {torch.__version__} |
            | Torchvision | {torchvision.__version__} |
            | Pre-training | COCO 2017 — 118K görüntü, 80 sınıf |
            """)

    print("\n🚀 Gradio arayüzü başlatılıyor...")
    print("⏳ \'Running on public URL\' linkini bekleyin")
    demo.launch(share=True, debug=False, show_error=True)


In [26]:
# ============================================================
# HÜCRE 9 — DEMO'YU BAŞLAT
# ============================================================
build_and_launch()


🚀 Gradio arayüzü başlatılıyor...
⏳ 'Running on public URL' linkini bekleyin
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9c67d990a8e1171663.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
